# Global Prochlorococcus Gene Prevalence Estimates

The purpose of this notebook is to generate a global estimate of the prevalence of every
known gene family in the Prochlorococcus pangenome.

In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from gene_prevalence import BernoulliPrevalenceModel

# instatiate random number generator with fixed seed for reproducibility
rng = np.random.default_rng(seed=9481)

# significance level for confidence intervals
ci_alpha = 0.05

In [2]:
# set aesthetic parameters

line_color = '#0F0A0A'    # dark

neutral_color = '#E0E0E0'

# thursday
accent_colors = ['#9B5DE5', '#FFAC69', '#00C9AE', '#FD3F92', '#0F0A0A', 
                 '#959AB1', '#FFDB66', '#63B9FF','#FFB1CA', '#4F1DD7']

style = {'axes.edgecolor': line_color,
         'axes.labelcolor': line_color,
         'text.color': line_color,
         'xtick.color': line_color,
         'ytick.color': line_color,
         'font.family': 'Helvetica',
         'font.Helvetica': ['Helvetica']}

palette = sns.color_palette(accent_colors)

sns.set_context('talk', rc={'lines.linewidth': 2})
sns.set_palette(palette)
sns.set_style('ticks', style)

plt.rcParams['legend.frameon'] = False
plt.rcParams['axes.spines.left'] = True
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.bottom'] = True
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['patch.linewidth'] = 0
plt.rcParams['patch.edgecolor'] = 'none'
plt.rcParams['savefig.dpi'] = 300

In [47]:
# import data

# directory paths
project_root = Path.cwd().resolve().parent
data_dir = project_root / "data" / "inputs"

# pro cell counts global climatology
pro_cell_counts_df = pd.read_csv(data_dir / "PRO_annual_climatological_mean.csv").rename(
    columns={"Latitude'": 'lat', "Longitude'": 'lon'}
)

# pro relative abundances by biome and clade
pro_rel_abund_df = pd.read_csv(data_dir / "Pro_Ecotype_Recipe_Rel_Abundance.csv")
pro_rel_abund_df['biome'] = pro_rel_abund_df['bin'].str.split('_').str[0]
pro_rel_abund_df['depth_bin'] = pro_rel_abund_df['bin'].str.split('_').str[1].map(
    {'upper50m': '0-50m', '50to100m': '50-100m', '100to200m': '100-200m'})

# lonhurst provinces shapefile
longhurst = gpd.read_file(data_dir / "longhurst_v4_2010/Longhurst_world_v4_2010.shp").to_crs("EPSG:4326")

longhurst

,ProvCode,ProvDescr,geometry
0,BPLR,Polar - Boreal Polar Province (POLR),"MULTIPOLYGON (((-161.18426 63.5, -161.5 63.5, ..."
1,ARCT,Polar - Atlantic Arctic Province,"MULTIPOLYGON (((-21.51305 64.64409, -21.55945 ..."
2,SARC,Polar - Atlantic Subarctic Province,"MULTIPOLYGON (((11.26472 63.96082, 11.09548 63..."
3,NADR,Westerlies - N. Atlantic Drift Province (WWDR),"POLYGON ((-11.5 57.5, -11.5 56.5, -11.5 55.5, ..."
4,GFST,Westerlies - Gulf Stream Province,"POLYGON ((-43.5 43.5, -43.5 42.5, -43.5 41.5, ..."
5,NASW,Westerlies - N. Atlantic Subtropical Gyral Pro...,"POLYGON ((-39.5 25.5, -40.5 25.5, -41.5 25.5, ..."
6,NATR,Trades - N. Atlantic Tropical Gyral Province (...,"MULTIPOLYGON (((-72.34673 18.53597, -72.36877 ..."
7,WTRA,Trades - Western Tropical Atlantic Province,"POLYGON ((-19.5 -6.5, -20.5 -6.5, -21.5 -6.5, ..."
8,ETRA,Trades - Eastern Tropical Atlantic Province,"POLYGON ((9.5 -12.5, 8.5 -12.5, 7.5 -12.5, 6.5..."
9,SATL,Trades - South Atlantic Gyral Province (SATG),"POLYGON ((-19.5 -6.5, -18.5 -6.5, -17.5 -6.5, ..."


In [40]:
# calculate multiplication factors for each depth range

# aggregate cell counts into three depth ranges: 0-50m, 50-100m, and 100-200m
pro_cell_counts_df['0-50m'] = pro_cell_counts_df.loc[:, "0m'":"50m'"].mean(axis=1)
pro_cell_counts_df['50-100m'] = pro_cell_counts_df.loc[:, "50m'":"100m'"].mean(axis=1)
# multiply by two to account for twice the volume in this depth range
pro_cell_counts_df['100-200m'] = pro_cell_counts_df.loc[:, "100m'":"200m'"].mean(axis=1) * 2

# downsample to only include rows with non-null values in at least one of the depth ranges
pro_agg_df = pro_cell_counts_df[pro_cell_counts_df[['0-50m', '50-100m', '100-200m']].notnull().any(
    axis=1)][['lat', 'lon', '0-50m', '50-100m', '100-200m']].reset_index(drop=True)

# join longhurst provinces with pro cell counts
gridded_pro_df = gpd.GeoDataFrame(
    pro_agg_df,
    geometry=gpd.points_from_xy(pro_agg_df['lon'], pro_agg_df['lat']),
    crs="EPSG:4326"
)

# attach province information to each point
# NOTE: within or intersects?
gridded_pro_df = gpd.sjoin(
    gridded_pro_df, longhurst, how='left', predicate='intersects')

# identify biome for each point based on province information
gridded_pro_df['biome'] = gridded_pro_df['ProvDescr'].str.split(' - ').str[0]

# aggregate by biome to get multiplication factor for each biome
pro_factors_df = gridded_pro_df.groupby('biome')[['0-50m', '50-100m', '100-200m']].sum()

# drop polar biome
pro_factors_df = pro_factors_df.loc[pro_factors_df.index != 'Polar']

# normalize by the minimum value across all biomes and depth ranges to get multiplication factors
pro_factors_df = pro_factors_df / pro_factors_df.min().min()

pro_factors_df

,0-50m,50-100m,100-200m
biome,,,
Coastal,2.352964,1.647873,1.000000
Trades,14.093870,12.402838,9.713393
Westerlies,6.094294,5.724519,6.418759


In [61]:
# calculate global relative abundances

# add on multiplication factors to pro relative abundances dataframe
global_rel_abund_df = pro_rel_abund_df.merge(
    pro_factors_df.reset_index().melt(var_name='depth_bin', id_vars='biome', value_name='factor'), 
    on=['biome', 'depth_bin'], 
    how='left'
)

# multiply relative abundances by multiplication factors to get global relative abundances
global_rel_abund_df['multiplied_rel_abundance'] = global_rel_abund_df['rel_abundance'] * global_rel_abund_df['factor']

# aggregate by clade to get global relative abundance for each clade
pro_global_rel_abund_df = global_rel_abund_df.groupby('clade')['multiplied_rel_abundance'].sum()
pro_global_rel_abund_df = pro_global_rel_abund_df / pro_global_rel_abund_df.sum()
pro_global_rel_abund_df = pro_global_rel_abund_df.reset_index().rename(
    columns={'multiplied_rel_abundance': 'global_rel_abundance'})

pro_global_rel_abund_df


,clade,global_rel_abundance
0,HL-I/VI,0.042424
1,HL-unclassified,0.021326
2,HL/LL,0.032114
3,HLI,0.175874
4,HLII,0.437426
5,HLVI,0.002432
6,LL-unclassified,0.040201
7,LLI,0.140869
8,LLII-III,0.009637
9,LLIV,0.008812
